In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure data handling
import pandas as pd

# Import the reusable CRUD Python module from Project One
from CRUD_Python_Module import AnimalShelter


###########################
# Data Manipulation / Model
###########################

# MongoDB credentials for the aacuser account.
# If your Module Three password is different, replace SNHU1234 below.
username = "aacuser"
password = "PinkOshi972"

# Connect to MongoDB through the CRUD module
db = AnimalShelter(username, password)


def records_to_dataframe(query):
    """Run a MongoDB query through the CRUD module and return a Dash-safe DataFrame."""
    records = db.read(query)
    dff = pd.DataFrame.from_records(records)

    # MongoDB ObjectId values are not JSON serializable by Dash.
    if "_id" in dff.columns:
        dff.drop(columns=["_id"], inplace=True)

    return dff


# An empty MongoDB query returns the unfiltered AAC Outcomes data set.
df = records_to_dataframe({})


#########################
# Dashboard Layout / View
#########################

app = JupyterDash(__name__)

# Grazioso Salvare logo
image_filename = "Grazioso Salvare Logo.png"
encoded_image = base64.b64encode(open(image_filename, "rb").read()).decode()


app.layout = html.Div([

    html.Center(
        html.A(
            html.Img(
                src="data:image/png;base64,{}".format(encoded_image),
                style={"height": "110px"}
            ),
            href="https://www.snhu.edu",
            target="_blank"
        )
    ),

    # Unique creator identifier
    html.Center(html.H1("Grazioso Salvare Rescue Animal Dashboard")),
    html.Center(html.H3("Created by Nahom Haile")),

    html.Hr(),

    # Interactive rescue-type filters
    html.Div([
        html.B("Rescue Type: "),
        dcc.RadioItems(
            id="filter-type",
            options=[
                {"label": "Water Rescue", "value": "Water Rescue"},
                {"label": "Mountain or Wilderness Rescue", "value": "Mountain or Wilderness Rescue"},
                {"label": "Disaster or Individual Tracking", "value": "Disaster or Individual Tracking"},
                {"label": "Reset", "value": "Reset"}
            ],
            value="Reset",
            inline=True,
            labelStyle={"marginRight": "20px"}
        )
    ]),

    html.Hr(),

    # Interactive animal data table
    dash_table.DataTable(
        id="datatable-id",
        columns=[
            {"name": column, "id": column, "deletable": False, "selectable": True}
            for column in df.columns
        ],
        data=df.to_dict("records"),

        # Single-row selection is required for the map callback.
        row_selectable="single",
        selected_rows=[0],

        # User-friendly table features
        filter_action="native",
        sort_action="native",
        sort_mode="multi",
        page_action="native",
        page_current=0,
        page_size=10,
        editable=False,
        row_deletable=False,

        style_table={"overflowX": "auto"},
        style_cell={
            "textAlign": "left",
            "minWidth": "100px",
            "width": "150px",
            "maxWidth": "200px",
            "whiteSpace": "normal"
        },
        style_header={"fontWeight": "bold"}
    ),

    html.Br(),
    html.Hr(),

    # Secondary chart and geolocation chart shown side by side
    html.Div(
        className="row",
        style={"display": "flex", "gap": "20px"},
        children=[
            html.Div(
                id="graph-id",
                className="col s12 m6",
                style={"width": "50%"}
            ),
            html.Div(
                id="map-id",
                className="col s12 m6",
                style={"width": "50%"}
            )
        ]
    )
])


#############################################
# Interaction Between Components / Controller
#############################################

@app.callback(
    [Output("datatable-id", "data"),
     Output("datatable-id", "selected_rows")],
    [Input("filter-type", "value")]
)
def update_dashboard(filter_type):
    """Query MongoDB for the rescue profile selected by the user."""

    if filter_type == "Water Rescue":
        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "Labrador Retriever Mix",
                    "Chesapeake Bay Retriever",
                    "Newfoundland"
                ]
            },
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }

    elif filter_type == "Mountain or Wilderness Rescue":
        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "German Shepherd",
                    "Alaskan Malamute",
                    "Old English Sheepdog",
                    "Siberian Husky",
                    "Rottweiler"
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }

    elif filter_type == "Disaster or Individual Tracking":
        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "Doberman Pinscher",
                    "German Shepherd",
                    "Golden Retriever",
                    "Bloodhound",
                    "Rottweiler"
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {
                "$gte": 20,
                "$lte": 300
            }
        }

    else:
        # Reset returns the dashboard to its original unfiltered state.
        query = {}

    filtered_df = records_to_dataframe(query)

    # Select the first returned animal so the map has an initial record.
    selected_rows = [0] if not filtered_df.empty else []

    return filtered_df.to_dict("records"), selected_rows


# Display a breed distribution chart based on the currently visible table data.
@app.callback(
    Output("graph-id", "children"),
    [Input("datatable-id", "derived_virtual_data")]
)
def update_graphs(viewData):

    if viewData is None:
        dff = df.copy()
    else:
        dff = pd.DataFrame.from_dict(viewData)

    if dff.empty or "breed" not in dff.columns:
        return html.P("No breed data is available for the selected filter.")

    # Group identical breeds and show the ten most common breeds.
    breed_counts = (
        dff["breed"]
        .fillna("Unknown")
        .value_counts()
        .head(10)
        .rename_axis("breed")
        .reset_index(name="count")
    )

    figure = px.pie(
        breed_counts,
        names="breed",
        values="count",
        title="Breed Distribution"
    )

    return dcc.Graph(figure=figure)


# Highlight selected DataTable columns.
@app.callback(
    Output("datatable-id", "style_data_conditional"),
    [Input("datatable-id", "selected_columns")]
)
def update_styles(selected_columns):

    if selected_columns is None:
        selected_columns = []

    return [
        {
            "if": {"column_id": column},
            "backgroundColor": "#D2F3FF"
        }
        for column in selected_columns
    ]


# Update the geolocation chart for the selected animal.
@app.callback(
    Output("map-id", "children"),
    [Input("datatable-id", "derived_virtual_data"),
     Input("datatable-id", "derived_virtual_selected_rows")]
)
def update_map(viewData, index):

    if viewData is None:
        dff = df.copy()
    else:
        dff = pd.DataFrame.from_dict(viewData)

    if dff.empty:
        return html.P("No animal location is available for the selected filter.")

    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    # If filtering changed the table and the old selection is no longer valid,
    # fall back to the first record.
    if row >= len(dff):
        row = 0

    animal = dff.iloc[row]

    latitude = animal.get("location_lat")
    longitude = animal.get("location_long")

    if pd.isna(latitude) or pd.isna(longitude):
        return html.P("Location information is unavailable for this animal.")

    return [
        dl.Map(
            style={"width": "100%", "height": "500px"},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(
                    position=[latitude, longitude],
                    children=[
                        dl.Tooltip(str(animal.get("breed", "Unknown Breed"))),
                        dl.Popup([
                            html.H3("Animal Name"),
                            html.P(str(animal.get("name", "Unknown")))
                        ])
                    ]
                )
            ]
        )
    ]


# Run app
app.run_server()


Dash app running on https://eddiecitizen-gondolarespect-3000.codio.io/proxy/8050/
